## PINN for Power System Transient Stability — Scenario 1 (Simulink Reference)

**Phase 2:** PINN vs Simulink comparison.

This notebook trains a PINN using the **Simulink simulation** of the IEEE 9-bus system as the
reference (supervised data anchor), rather than the RK4 solution used in Phase 1.

**Scenario 1:** Three-phase fault at Bus 7, cleared at 5 cycles (5/60 s) by opening line 5–7.

The PINN physics loss still uses the classical swing equations (same as Phase 1). The data loss
is anchored to the Simulink trajectory, which uses a full d-q machine model — a more realistic
reference. This tests whether the classical-physics PINN can fit a higher-fidelity reference.

In [ ]:
import math
import os
import numpy as np
import torch
import torch.nn as nn
import torch.autograd as autograd
import matplotlib.pyplot as plt
import pandas as pd
import scipy.io

# CONFIGURATION AND HYPERPARAMETERS

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print(f'Available GPUs: {torch.cuda.device_count()}')
if torch.cuda.device_count() > 1:
    print(f'GPU devices: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}')

os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

# Simulation parameters
f              = 60.0
t_fault_clear  = 5.0 / 60.0   # 5 cycles at 60 Hz
t_end          = 2.0

# Network architecture
NETWORK_WIDTH      = 512
NETWORK_DEPTH      = 8
LEARNING_RATE      = 5e-4
SCHEDULER_PATIENCE = 500

# Training data
N_C_BASE  = 16000   # base random collocation points
N_C_FAULT = 16000   # dense points around fault clearing
N_C_EARLY = 8000    # dense points in early transient
N_IC      = 256     # initial condition points
N_DATA    = 2000    # supervised Simulink anchor points

# Loss weights
PDE_LOSS_WEIGHT  = 0.5
IC_LOSS_WEIGHT   = 1.0
DATA_LOSS_WEIGHT = 5.0

# Training
NUM_ITERS  = 35000
PRINT_FREQ = 200
CHECK_FREQ = 1000

print(f"{'='*70}")
print('HYPERPARAMETER SUMMARY — SCENARIO 1: SIMULINK REFERENCE')
print(f"{'='*70}")
print(f'Collocation points: {N_C_BASE + N_C_FAULT + N_C_EARLY:,} ({N_C_BASE}+{N_C_FAULT}+{N_C_EARLY})')
print(f'Training iterations: {NUM_ITERS:,}')
print(f'Network: {NETWORK_WIDTH}×{NETWORK_DEPTH}')
print(f'Loss weights — PDE: {PDE_LOSS_WEIGHT} | IC: {IC_LOSS_WEIGHT} | Data: {DATA_LOSS_WEIGHT}')
print(f'Data anchor points (Simulink): {N_DATA}')
print(f"{'='*70}")

# SYSTEM PARAMETERS

Physics parameters are identical to Phase 1 (pinn-scenario-1-final.ipynb).
Source: Anderson & Fouad, IEEE 9-bus system, 100 MVA base.

In [ ]:
w_s = 2 * math.pi * f

# Inertia constants (on 100 MVA system base)
# H_sys = H_machine * (MVA_machine / 100)
#   G1: 9.55 * (247.5/100) = 23.64  (stored energy 2364 MJ / 247.5 MVA)
#   G2: 3.33 * (192.0/100) = 6.40   (stored energy  640 MJ / 192   MVA)
#   G3: 2.35 * (128.0/100) = 3.01   (stored energy  301 MJ / 128   MVA)
H  = torch.tensor([23.64, 6.40, 3.01], dtype=torch.float32)
D  = torch.tensor([0.0,   0.0,  0.0],  dtype=torch.float32)  # no damping

# Internal voltage magnitudes (from load flow, Anderson & Fouad Eq. 2.6)
E  = torch.tensor([1.0566, 1.0502, 1.0170], dtype=torch.float32)

# Mechanical power (on 100 MVA base: 71.6/100, 163/100, 85/100)
# Note: Pm below is the INITIAL guess only; it is overwritten from Simulink ICs in the
# next cell (Pe(delta0_sim, Y_clear) = Pm ensures zero swing residual at t=0).
Pm = torch.tensor([0.716, 1.63, 0.85], dtype=torch.float32)

# Initial conditions — placeholder; overwritten from .mat file below
delta0_deg = torch.tensor([2.2717, 19.7315, 13.1752], dtype=torch.float32)
delta0     = torch.deg2rad(delta0_deg)
omega0     = torch.zeros(3, dtype=torch.float32)

M = 2 * H / w_s   # angular momentum M = 2H / omega_s

# ---- Reduced Y matrices (Anderson & Fouad, Tables 2.4 and 2.5) ----
# Same matrices used in Phase 1.  Rows/cols: [G1, G2, G3]

def Y_from_entries(entries):
    Y = torch.zeros((3, 3), dtype=torch.complex64)
    for (i, j, gr, gi) in entries:
        Y[i, j] = torch.complex(
            torch.tensor(gr, dtype=torch.float32),
            torch.tensor(gi, dtype=torch.float32))
    return Y

# During fault (three-phase fault at Bus 7)
Y_fault = Y_from_entries([
    (0,0,  0.657, -3.816), (0,1,  0.000,  0.000), (0,2,  0.070,  0.631),
    (1,0,  0.000,  0.000), (1,1,  0.000, -5.486), (1,2,  0.000,  0.000),
    (2,0,  0.070,  0.631), (2,1,  0.000,  0.000), (2,2,  0.174, -2.796),
])

# After fault clearing (line 5-7 opened)
Y_clear = Y_from_entries([
    (0,0,  1.181, -2.229), (0,1,  0.138,  0.726), (0,2,  0.191,  1.079),
    (1,0,  0.138,  0.726), (1,1,  0.389, -1.953), (1,2,  0.199,  1.229),
    (2,0,  0.191,  1.079), (2,1,  0.199,  1.229), (2,2,  0.273, -2.342),
])

def piecewise_Y(t):
    """Return the 3x3 Y matrix as a function of time."""
    Y1  = Y_fault.to(t.device)
    Y2  = Y_clear.to(t.device)
    out = torch.zeros((t.shape[0], 3, 3), dtype=torch.complex64, device=t.device)
    out[t.squeeze() <  t_fault_clear] = Y1
    out[t.squeeze() >= t_fault_clear] = Y2
    return out

def electrical_power(delta, Evec, Ybatch):
    """Compute electrical output power Pe for each generator."""
    Evec = Evec.to(delta.device)
    Ecol = Evec.view(1, 3)
    dmat = delta.unsqueeze(2) - delta.unsqueeze(1)
    G    = Ybatch.real
    B    = Ybatch.imag
    term = G * torch.cos(dmat) + B * torch.sin(dmat)
    EE   = Ecol.unsqueeze(2) * Ecol.unsqueeze(1)
    return torch.sum(EE * term, dim=2)

print('System parameters loaded.')
print(f'  H  = {H.tolist()}')
print(f'  E  = {E.tolist()}')
print(f'  Pm = {Pm.tolist()} (will be recomputed from Simulink ICs)')

# REFERENCE SOLUTION — SIMULINK

Load the Simulink trajectory exported by `matlab/run_scenarios.m` (scenario = 1).
The .mat file contains delta and omega interpolated to a 4001-point uniform grid.

We also recompute `Pm` from the Simulink initial state so that the swing-equation
residual is exactly zero at t=0, preventing physics loss from fighting data loss.

In [ ]:
print('Loading Simulink reference for Scenario 1...')

# Update this path to wherever you uploaded the .mat file (e.g. Kaggle input)
MAT_PATH = '/kaggle/input/simulink-outputs/scenario_1_simulink_outputs.mat'

mat = scipy.io.loadmat(MAT_PATH)

T_ref     = mat['T_ref'].astype('float32').squeeze()   # (4001,)
delta_ref = mat['delta_sim_i'].astype('float32')       # (4001, 3) radians
omega_ref = mat['omega_sim_i'].astype('float32')       # (4001, 3) rad/s

print(f'T_ref shape   : {T_ref.shape}')
print(f'delta_ref shape: {delta_ref.shape}')
print(f'omega_ref shape: {omega_ref.shape}')
print(f'Time range     : {T_ref[0]:.4f} s  to  {T_ref[-1]:.4f} s')
print(f'delta_ref[0] in degrees: {np.rad2deg(delta_ref[0])}')

# Load explicit initial state exported by run_scenarios.m
if 'delta0_sim' in mat:
    delta0 = torch.tensor(mat['delta0_sim'].flatten(), dtype=torch.float32)
    omega0 = torch.tensor(mat['omega0_sim'].flatten(), dtype=torch.float32)
else:
    delta0 = torch.tensor(delta_ref[0], dtype=torch.float32)
    omega0 = torch.tensor(omega_ref[0], dtype=torch.float32)

# Recompute Pm so Pe(delta0, Y_clear) = Pm (zero residual at t=0)
with torch.no_grad():
    _d0  = delta0.unsqueeze(0)            # (1, 3)
    _Y0  = Y_clear.unsqueeze(0)           # (1, 3, 3)
    _Pe0 = electrical_power(_d0, E, _Y0)  # (1, 3)
    Pm   = _Pe0.squeeze(0).cpu()          # (3,)

print(f'\nInitial conditions from Simulink:')
print(f'  delta0  (deg) : {np.rad2deg(delta0.numpy())}')
print(f'  omega0 (rad/s): {omega0.numpy()}')
print(f'  Pm recomputed : {Pm.numpy()}')
print(f'  Pm original   : [0.716, 1.630, 0.850]')

# PINN NEURAL NETWORK

In [ ]:
class PINN(nn.Module):
    """Physics-Informed Neural Network for the IEEE 9-bus swing equations.
    
    Architecture: 8-layer MLP with 512 hidden units and Tanh activations.
    Output transformation hard-encodes the initial conditions:
        delta(t) = delta0 + t * NN_delta(t)
        omega(t) = omega0 + t * NN_omega(t)
    """
    def __init__(self, width=512, depth=8):
        super().__init__()
        layers = []
        inp = 1
        for i in range(depth):
            layers += [nn.Linear(inp, width)]
            if i < depth - 1:
                layers += [nn.Tanh()]
            inp = width
        self.body = nn.Sequential(*layers)
        self.out  = nn.Linear(width, 6)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight, gain=1.0)
                nn.init.zeros_(m.bias)

    def forward(self, t):
        t_norm = t / t_end
        z = self.body(t_norm)
        y = self.out(z)
        delta = delta0.to(t.device).unsqueeze(0) + t * y[:, :3]
        omega = omega0.to(t.device).unsqueeze(0) + t * y[:, 3:]
        return delta, omega

torch.manual_seed(42)
net = PINN(width=NETWORK_WIDTH, depth=NETWORK_DEPTH).to(device)

if torch.cuda.device_count() > 1:
    print(f'Using DataParallel with {torch.cuda.device_count()} GPUs')
    net = nn.DataParallel(net)

opt = torch.optim.Adam(net.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt, mode='min', factor=0.5, patience=SCHEDULER_PATIENCE
)

total_params = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(f'PINN parameters: {total_params:,}')

# TRAINING DATA

In [ ]:
print('Setting up training data...')

# Collocation points for PDE residual
t_c_rand  = torch.rand(N_C_BASE, 1, device=device) * t_end
extra_fault = torch.linspace(
    max(0.0, t_fault_clear - 0.03), min(t_end, t_fault_clear + 0.03),
    N_C_FAULT, device=device).unsqueeze(1)
extra_early = torch.linspace(0.0, 0.2, N_C_EARLY, device=device).unsqueeze(1)
t_c = torch.cat([t_c_rand, extra_fault, extra_early], dim=0)
t_c = t_c[torch.randperm(t_c.shape[0])]
print(f'Total collocation points: {t_c.shape[0]:,}')

# Initial condition constraint points (t = 0)
t_ic = torch.zeros((N_IC, 1), dtype=torch.float32, device=device)

# Supervised data anchors — uniformly sampled from the Simulink trajectory
idx        = np.linspace(0, len(T_ref) - 1, N_DATA, dtype=int)
t_data     = torch.tensor(T_ref[idx],        dtype=torch.float32, device=device).view(-1, 1)
delta_data = torch.tensor(delta_ref[idx],    dtype=torch.float32, device=device)
omega_data = torch.tensor(omega_ref[idx],    dtype=torch.float32, device=device)

# Cache device tensors
delta0_d = delta0.to(device)
omega0_d = omega0.to(device)
M_d      = M.to(device)
Pm_d     = Pm.to(device)
D_d      = D.to(device)
E_d      = E.to(device)

print(f'Simulink data anchors: {N_DATA} points uniformly sampled from {len(T_ref)} Simulink steps')

# LOSS FUNCTION

Three-component loss:
1. **PDE loss** — classical swing-equation residuals (same physics as Phase 1 RK4 PINN)
2. **IC loss** — zero because ICs are hard-encoded in the network output layer
3. **Data loss** — MSE between PINN output and Simulink trajectories

In [ ]:
def pinn_loss():
    t = t_c.requires_grad_(True)
    delta, omega = net(t)

    # Automatic differentiation for time derivatives
    ddelta_dt = torch.zeros_like(delta)
    domega_dt = torch.zeros_like(omega)
    for i in range(3):
        ddelta_dt[:, i:i+1] = autograd.grad(
            delta[:, i:i+1], t, torch.ones_like(delta[:, i:i+1]),
            retain_graph=True, create_graph=True)[0]
        domega_dt[:, i:i+1] = autograd.grad(
            omega[:, i:i+1], t, torch.ones_like(omega[:, i:i+1]),
            retain_graph=True, create_graph=True)[0]

    # Electrical power from swing-equation physics
    Yb = piecewise_Y(t.squeeze())
    Pe = electrical_power(delta, E_d, Yb)

    # Swing equation residuals
    kin_res = ddelta_dt - omega
    dyn_res = domega_dt - (
        (Pm_d.unsqueeze(0) - Pe) / M_d.unsqueeze(0)
        - (D_d.unsqueeze(0) * omega) / M_d.unsqueeze(0)
    )
    loss_pde = torch.mean(kin_res**2) + torch.mean(dyn_res**2)

    # IC loss (should be ~0 due to hard encoding)
    d0, w0 = net(t_ic)
    ic_loss = torch.mean((d0 - delta0_d)**2) + torch.mean((w0 - omega0_d)**2)

    # Data loss — fit Simulink trajectories
    dD, wD   = net(t_data)
    data_loss = DATA_LOSS_WEIGHT * (
        torch.mean((dD - delta_data)**2) + torch.mean((wD - omega_data)**2)
    )

    total = PDE_LOSS_WEIGHT * loss_pde + IC_LOSS_WEIGHT * ic_loss + data_loss
    return total, (loss_pde.item(), ic_loss.item(), data_loss.item())

# TRAINING

In [ ]:
print(f"{'='*70}")
print('TRAINING PINN — SCENARIO 1: SIMULINK REFERENCE')
print(f"{'='*70}\n")

history   = []
best_loss = float('inf')

for it in range(NUM_ITERS):
    opt.zero_grad()
    L, parts = pinn_loss()
    L.backward()
    torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
    opt.step()
    scheduler.step(L)

    if L.item() < best_loss:
        best_loss = L.item()

    if (it + 1) % PRINT_FREQ == 0:
        history.append((it + 1, float(L), parts))
        print(f'iter {it+1:5d}: total={L:.4e}  pde={parts[0]:.4e}  '
              f'ic={parts[1]:.4e}  data={parts[2]:.4e}')

        if (it + 1) % CHECK_FREQ == 0:
            with torch.no_grad():
                t_chk = torch.linspace(0, t_end, 100, device=device).view(-1, 1)
                d_chk, w_chk = net(t_chk)
                print(f'  -> delta range: [{d_chk.min():.2f}, {d_chk.max():.2f}] rad '
                      f'| omega range: [{w_chk.min():.2f}, {w_chk.max():.2f}] rad/s')

# EVALUATION

In [ ]:
print(f"\n{'='*70}")
print('EVALUATION')
print(f"{'='*70}\n")

net_module = net.module if isinstance(net, nn.DataParallel) else net

t_eval = torch.tensor(T_ref, dtype=torch.float32, device=device).view(-1, 1)
with torch.no_grad():
    delta_pinn, omega_pinn = net_module(t_eval)

delta_pinn = delta_pinn.cpu().numpy()
omega_pinn = omega_pinn.cpu().numpy()

err_deg        = np.rad2deg(np.abs(delta_pinn - delta_ref))
max_error_deg  = float(err_deg.max())
mean_error_deg = float(err_deg.mean())

print(f'Max error  (vs Simulink): {max_error_deg:.4f} deg')
print(f'Mean error (vs Simulink): {mean_error_deg:.4f} deg')

# SAVE RESULTS

In [ ]:
out_dir = 'scenario_1_simulink_outputs'
os.makedirs(out_dir, exist_ok=True)

np.savez(f'{out_dir}/reference_outputs.npz',
         T_ref=T_ref, delta_ref=delta_ref, omega_ref=omega_ref)
np.savez(f'{out_dir}/pinn_outputs.npz',
         T_eval=T_ref, delta_pinn=delta_pinn, omega_pinn=omega_pinn)
np.savez(f'{out_dir}/summary_metrics.npz',
         max_error_deg=np.array([max_error_deg]),
         mean_error_deg=np.array([mean_error_deg]))

results_df = pd.DataFrame({
    'Time (s)': T_ref,
    'delta_1_Simulink (deg)': np.rad2deg(delta_ref[:, 0]),
    'delta_2_Simulink (deg)': np.rad2deg(delta_ref[:, 1]),
    'delta_3_Simulink (deg)': np.rad2deg(delta_ref[:, 2]),
    'delta_1_PINN (deg)': np.rad2deg(delta_pinn[:, 0]),
    'delta_2_PINN (deg)': np.rad2deg(delta_pinn[:, 1]),
    'delta_3_PINN (deg)': np.rad2deg(delta_pinn[:, 2]),
    'omega_1_Simulink (rad/s)': omega_ref[:, 0],
    'omega_2_Simulink (rad/s)': omega_ref[:, 1],
    'omega_3_Simulink (rad/s)': omega_ref[:, 2],
    'omega_1_PINN (rad/s)': omega_pinn[:, 0],
    'omega_2_PINN (rad/s)': omega_pinn[:, 1],
    'omega_3_PINN (rad/s)': omega_pinn[:, 2],
    'error_delta_1 (deg)': err_deg[:, 0],
    'error_delta_2 (deg)': err_deg[:, 1],
    'error_delta_3 (deg)': err_deg[:, 2],
    'max_error (deg)': err_deg.max(axis=1),
})
results_df.to_csv(f'{out_dir}/results_detailed.csv', index=False)

with open(f'{out_dir}/summary_statistics.txt', 'w') as fout:
    fout.write('='*70 + '\n')
    fout.write('SCENARIO 1 — Three-Phase Fault at Bus 7 (SIMULINK REFERENCE)\n')
    fout.write('='*70 + '\n\n')
    fout.write('PINN CONFIGURATION:\n')
    fout.write(f'  Network : {NETWORK_WIDTH}x{NETWORK_DEPTH} Tanh, Xavier Normal\n')
    fout.write(f'  Collocation : {t_c.shape[0]:,} points\n')
    fout.write(f'  Iterations  : {NUM_ITERS:,}\n')
    fout.write(f'  PDE weight  : {PDE_LOSS_WEIGHT}\n')
    fout.write(f'  Data weight : {DATA_LOSS_WEIGHT}\n\n')
    fout.write('RESULTS:\n')
    fout.write(f'  Max error  (vs Simulink): {max_error_deg:.4f} deg\n')
    fout.write(f'  Mean error (vs Simulink): {mean_error_deg:.4f} deg\n')
    fout.write(f'  Median error           : {np.median(err_deg):.4f} deg\n')
    fout.write(f'  Std dev error          : {np.std(err_deg):.4f} deg\n\n')
    fout.write('PER-GENERATOR ERRORS:\n')
    for i in range(3):
        fout.write(f'  G{i+1}: max={err_deg[:,i].max():.4f}  '
                   f'mean={err_deg[:,i].mean():.4f}  '
                   f'std={np.std(err_deg[:,i]):.4f} deg\n')

print(f'Saved to: {out_dir}/')
print(f'max_error_deg={max_error_deg:.4f}, mean_error_deg={mean_error_deg:.4f}')

# PLOTTING

In [ ]:
print(f"{'='*70}")
print('GENERATING PLOTS')
print(f"{'='*70}\n")

# 1. Simulink reference
plt.figure(figsize=(10, 6))
for i, (lbl, ls) in enumerate(zip([r'$\delta_1$',r'$\delta_2$',r'$\delta_3$'],
                                  ['-','--','-.'])):
    plt.plot(T_ref, np.rad2deg(delta_ref[:,i]), ls, linewidth=2, label=lbl)
plt.axvline(t_fault_clear, linestyle=':', linewidth=2, color='red', label='Fault cleared')
plt.xlabel('Time (s)', fontsize=12); plt.ylabel('Rotor Angle (deg)', fontsize=12)
plt.title('Simulink Reference — Scenario 1 (Three-Phase Fault at Bus 7)', fontweight='bold')
plt.grid(True, alpha=0.3); plt.legend(fontsize=11); plt.tight_layout()
plt.savefig(f'{out_dir}/01_delta_simulink.png', dpi=300, bbox_inches='tight'); plt.show()

# 2. PINN output
plt.figure(figsize=(10, 6))
for i, (lbl, ls) in enumerate(zip([r'$\delta_1$',r'$\delta_2$',r'$\delta_3$'],
                                  ['-','--','-.'])):
    plt.plot(T_ref, np.rad2deg(delta_pinn[:,i]), ls, linewidth=2, label=lbl)
plt.axvline(t_fault_clear, linestyle=':', linewidth=2, color='red', label='Fault cleared')
plt.xlabel('Time (s)', fontsize=12); plt.ylabel('Rotor Angle (deg)', fontsize=12)
plt.title('PINN Output — Scenario 1', fontweight='bold')
plt.grid(True, alpha=0.3); plt.legend(fontsize=11); plt.tight_layout()
plt.savefig(f'{out_dir}/02_delta_pinn.png', dpi=300, bbox_inches='tight'); plt.show()

# 3. PINN vs Simulink comparison
plt.figure(figsize=(12, 7))
colors = ['tab:blue', 'tab:orange', 'tab:green']
for i, (lbl, c) in enumerate(zip([r'$\delta_1$',r'$\delta_2$',r'$\delta_3$'], colors)):
    plt.plot(T_ref, np.rad2deg(delta_ref[:,i]),  '-',  color=c, alpha=0.7,
             linewidth=2.5, label=f'{lbl} (Simulink)')
    plt.plot(T_ref, np.rad2deg(delta_pinn[:,i]), '--', color=c,
             linewidth=2,   label=f'{lbl} (PINN)')
plt.axvline(t_fault_clear, linestyle=':', linewidth=2, color='red', label='Fault cleared')
plt.xlabel('Time (s)', fontsize=12); plt.ylabel('Rotor Angle (deg)', fontsize=12)
plt.title('PINN vs Simulink — Scenario 1', fontweight='bold', fontsize=14)
plt.grid(True, alpha=0.3); plt.legend(fontsize=10, ncol=2); plt.tight_layout()
plt.savefig(f'{out_dir}/03_comparison.png', dpi=300, bbox_inches='tight'); plt.show()

# 4. Error analysis
plt.figure(figsize=(12, 7))
for i, (lbl, c) in enumerate(zip([r'$|\Delta\delta_1|$',r'$|\Delta\delta_2|$',r'$|\Delta\delta_3|$'],
                                  colors)):
    plt.plot(T_ref, err_deg[:,i], linewidth=2.5, color=c, label=lbl)
plt.plot(T_ref, err_deg.max(axis=1), 'k--', linewidth=3, label='Max error')
plt.axvline(t_fault_clear, linestyle=':', linewidth=2, color='red', alpha=0.7)
plt.fill_between(T_ref, 0, err_deg.max(axis=1), alpha=0.1, color='gray')
plt.xlabel('Time (s)', fontsize=12); plt.ylabel('|Δδ| (deg)', fontsize=12)
plt.title('PINN Prediction Error vs Simulink — Scenario 1', fontweight='bold', fontsize=14)
plt.grid(True, alpha=0.3); plt.legend(fontsize=11); plt.tight_layout()
plt.savefig(f'{out_dir}/04_error_analysis.png', dpi=300, bbox_inches='tight'); plt.show()

# 5. Angular velocities + per-generator error bars
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i in range(3):
    axes[0].plot(T_ref, omega_ref[:,i],  '-',  alpha=0.5, linewidth=2,
                 label=f'G{i+1} (Simulink)')
    axes[0].plot(T_ref, omega_pinn[:,i], '--', linewidth=2, label=f'G{i+1} (PINN)')
axes[0].axvline(t_fault_clear, linestyle=':', linewidth=2, color='red', alpha=0.7)
axes[0].set_xlabel('Time (s)', fontsize=11)
axes[0].set_ylabel('Angular Velocity (rad/s)', fontsize=11)
axes[0].set_title('Angular Velocities: PINN vs Simulink', fontweight='bold')
axes[0].grid(True, alpha=0.3); axes[0].legend(fontsize=9)
gen_labels = ['G1', 'G2', 'G3']
axes[1].bar(gen_labels, [err_deg[:,i].max()  for i in range(3)], label='Max error',  alpha=0.7)
axes[1].bar(gen_labels, [err_deg[:,i].mean() for i in range(3)], label='Mean error', alpha=0.7)
axes[1].set_ylabel('Error (deg)', fontsize=11)
axes[1].set_title('Error Metrics by Generator', fontweight='bold')
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f'{out_dir}/05_velocities_and_errors.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n{'='*70}")
print('SUMMARY')
print(f"{'='*70}")
print(f'Output directory : {out_dir}/')
print(f'Max angle error  : {max_error_deg:.4f} deg')
print(f'Mean angle error : {mean_error_deg:.4f} deg')
print(f"{'='*70}")